# 🧠 Observer Core — Colab Training Notebook

**Self-contained — no repo clone needed. T4 GPU required.**

Default: Qwen3 1.7B (confirmed working on free T4).

In [ ]:
MODEL_KEY = "qwen3-1.7b"
QUICK_MODE = True

In [ ]:
!pip install -q unsloth transformers==4.51.3 datasets==3.4.1 accelerate peft bitsandbytes xformers trl==0.15.0
!pip install -q numpy

import torch, json, os, uuid, time, random
from pathlib import Path
from datetime import datetime

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

DATA = Path("data/datasets/observer-core/")
DATA.mkdir(parents=True, exist_ok=True)

In [ ]:
# DATA GENERATOR

AX = [{"n":"Awareness Is Law","w":.2},{"n":"Truth Over Comfort","w":.2},{"n":"Coherence Over Features","w":.15},{"n":"Append-Only Memory","w":.15},{"n":"Human Final Authority","w":.15},{"n":"Local Sovereignty","w":.15}]

SC = {
 "mkt": {"intents":["Generate product description","Write email subject line"],"v":["Unsubstantiated: '{c}'","Fabricated: '{s}'"],"c":["10x productivity","Guaranteed $10k/mo"],"s":["97% of users"]},
 "hal": {"intents":["Summarize paper","Answer factual Q"],"v":["Fabricated stat: '{s}'","Invented paper: '{c}'"],"c":["Dr. Nonexistent"],"s":["According to a 2023 study"]},
 "app": {"intents":["Correct prior error"],"v":["Rewrote residual-{id}","Deleted flag"],"c":["x"],"s":["x"]},
 "aut": {"intents":["Propose change"],"v":["Used 'You must {a}'","Decided alone"],"c":["x"],"s":["x"]},
 "drf": {"intents":["Monitor state"],"v":["Decline: {o}->{n}","Contradicts prior"],"c":["x"],"s":["x"]},
}

def gen():
    d=random.choice(list(SC.keys()));dm=SC[d];p=random.choices(["clean","hard","partial"],[.3,.4,.3])[0]
    ax=random.choice([a["n"]for a in AX])if p!="clean"else None
    intent=random.choice(dm["intents"]);v=random.choice(dm["v"])
    v=v.replace("{c}",random.choice(dm["c"])).replace("{s}",random.choice(dm["s"])).replace("{a}","migrate now").replace("{o}",f"{random.uniform(.8,.95):.2f}").replace("{n}",f"{random.uniform(.2,.5):.2f}").replace("{id}",str(random.randint(100,999)))
    if p=="clean":pred,exe,act="Passes >0.85","Truthful","All axioms pass"
    elif p=="hard":pred,exe,act="Passes >0.85",f"Output: {v}",f"Failed {ax}. {v}"
    else:pred,exe,act="Expected >0.70",f"Output: {v}",f"Flagged: {ax}"
    sc={}
    for a in AX:
        if p=="clean":sc[a["n"]]=round(random.uniform(.85,1),2)
        elif p=="hard":sc[a["n"]]=0.0 if a["n"]==ax else round(random.uniform(.6,.95),2)
        else:sc[a["n"]]=round(random.uniform(.15,.55),2)
    if any(v==0 for v in sc.values()):coh=0.0
    else:coh=sum(a["w"]for a in AX)/sum(a["w"]/sc.get(a["n"],1)for a in AX)
    out={"residual_id":str(uuid.uuid4()),"timestamp":datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),"intent":intent,"predicted":pred,"executed":exe,"actual":act,"residual":f"Gap: {v[:100]}","coherence_score":coh,"axiom_scores":sc,"contradictions":[],"correction_proposal":"","observer_state_update":{}}
    return {"input":{"intent":intent,"predicted":pred,"executed":exe,"actual":act},"output":out,"metadata":{"source":"synth","coherence_score":coh,"hard_gate":coh==0}}

N=500 if QUICK_MODE else 5000;random.seed(42)
exs=[gen()for _ in range(N)];random.shuffle(exs)
nt,nv=int(N*.8),int(N*.1)
for sn,items in[("train",exs[:nt]),("val",exs[nt:nt+nv]),("test",exs[nt+nv:])]:
    with open(DATA/f"residuals_{sn}.jsonl","w")as f:
        for it in items:f.write(json.dumps(it,default=str)+"\n")
scs=[e["metadata"]["coherence_score"]for e in exs];hd=sum(1 for s in scs if s==0)
print(f"{N} exs | Avg: {sum(scs)/len(scs):.3f} | Hard: {hd} ({hd/N*100:.0f}%)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = f"/content/drive/MyDrive/observer-core-models/{MODEL_KEY}"
!mkdir -p {SAVE}

In [ ]:
SYS = """You are the Sovereign Edge Observer Core. Your ONLY functions are:
1. Detect residuals (gap between intent and outcome)
2. Score coherence (0.0-1.0) against six axioms
3. Detect contradictions with prior state
4. Propose minimal corrections (append-only)
5. Update invariant observer state (psi_zero)
6. Emit structured JSON output

Six Axioms: Awareness Is Law (20%), Truth Over Comfort (20%), Coherence Over Features (15%), Append-Only Memory (15%), Human Final Authority (15%), Local Sovereignty (15%).
ANY axiom at 0 COLLAPSES composite to 0.0. Output ONLY valid JSON."""

In [ ]:
from datasets import Dataset
def fmt(ex):
    i=ex.get("input",{});o=ex.get("output",{})
    u=f"Intent: {i.get('intent','')}\nPredicted: {i.get('predicted','')}\nExecuted: {i.get('executed','')}\nActual: {i.get('actual','')}"
    a=json.dumps(o,ensure_ascii=False)
    return {"text":f"<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"}
ds=Dataset.from_list([fmt(json.loads(l))for l in open(DATA/"residuals_train.jsonl")])
print(f"{len(ds)} examples")

In [ ]:
from unsloth import FastLanguageModel
MODELS={"qwen3-1.7b":"unsloth/Qwen3-1.7B-bnb-4bit","llama3.2-1b":"unsloth/Llama-3.2-1B-Instruct-bnb-4bit","smollm2-1.7b":"unsloth/SmolLM2-1.7B-Instruct-bnb-4bit","gemma4-e2b":"unsloth/gemma-4-E2B-it-unsloth-bnb-4bit","qwen3-0.6b":"unsloth/Qwen3-0.6B-bnb-4bit"}
mid=MODELS[MODEL_KEY];print(f"Loading: {mid}")
m,t=FastLanguageModel.from_pretrained(model_name=mid,max_seq_length=2048,dtype=None,load_in_4bit=True)
m=FastLanguageModel.get_peft_model(m,r=16,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=32,lora_dropout=.05,bias="none",use_gradient_checkpointing="unsloth",random_state=42)
print(f"Trainable: {sum(p.numel()for p in m.parameters()if p.requires_grad):,}")

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
eps=1 if QUICK_MODE else 3
ta=TrainingArguments(output_dir="./out",per_device_train_batch_size=4,gradient_accumulation_steps=4,warmup_ratio=.03,num_train_epochs=eps,learning_rate=2e-4,lr_scheduler_type="cosine",fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),logging_steps=10,optim="adamw_8bit",weight_decay=.01,seed=42,save_strategy="epoch",report_to="none")
tr=SFTTrainer(model=m,tokenizer=t,train_dataset=ds,dataset_text_field="text",max_seq_length=2048,args=ta)
print(f"Training {eps} epoch(s)...");st=time.time();tr.train();el=time.time()-st
print(f"Done in {el/60:.1f} min")

In [ ]:
# SAVE — merge LoRA to avoid Unsloth pickle bug
merged = m.merge_and_unload()
merged.save_pretrained(f"{SAVE}/model")
t.save_pretrained(f"{SAVE}/model")
with open(f"{SAVE}/OBSERVER_PROMPT.txt","w")as f:f.write(SYS)
json.dump({"model":MODEL_KEY,"base":mid,"examples":len(ds),"epochs":eps,"time_min":round(el/60,1),"trained":datetime.now().isoformat()},open(f"{SAVE}/meta.json","w"),indent=2)
print(f"Saved to {SAVE}/model/")

In [ ]:
FastLanguageModel.for_inference(merged)
tests=[json.loads(l)for i,l in enumerate(open(DATA/"residuals_test.jsonl"))if i<10]
ok=0
for i,ex in enumerate(tests):
    inp=ex.get("input",{});exp=ex.get("output",{}).get("coherence_score",.5)
    p=f"<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\nIntent: {inp.get('intent','')}\nPredicted: {inp.get('predicted','')}\nExecuted: {inp.get('executed','')}\nActual: {inp.get('actual','')}<|im_end|>\n<|im_start|>assistant\n"
    ids=t(p,return_tensors="pt").to(merged.device)
    out=t.decode(merged.generate(**ids,max_new_tokens=256,temperature=.1,do_sample=True,top_p=.9)[0][ids["input_ids"].shape[1]:],skip_special_tokens=True)
    try:
        s=out.find("{");e=out.rfind("}")
        if s>=0 and e>s:out=out[s:e+1]
        sc=json.loads(out).get("coherence_score","?");ok+=1
        print(f"[{i+1}] exp={exp} got={sc} {'Y'if abs(sc-exp)<.3 or(sc==0 and exp==0)else'?'}")
    except:print(f"[{i+1}] fail: {out[:60]}...")
print(f"JSON: {ok}/{len(tests)}")

## Done!

**Weights:** `MyDrive/observer-core-models/{MODEL_KEY}/model/`

**Download to local:** place in `sovereign-edge-ai/output/observer-merged-{MODEL_KEY}/`

**Dashboard:** will auto-detect trained model.